# 03 — Explanation Faithfulness (D7–D9)

**This notebook is the paper.** Detection accuracy is table stakes — ≥5 YOLO-family works already exist on VinDr-CXR. What no verified prior work does is *quantify* whether the model looks where the radiologist looked.

The asset: **VinDr ships ground-truth boxes.** Most CXR-XAI work uses NIH ChestX-ray14, which has none and can therefore only show qualitative heatmaps.

> ⚠️ **The trap:** "we ran three YOLOs and made heatmaps" is a course project — literally what the vendored `tariqshaban` repo already is. The measured saliency evaluation is the whole difference.
>
> **If the schedule slips, cut a model. Never cut these metrics.**

---
### Multi-seed here too — and it matters more here

These metrics carry the paper's claim, so they need error bars at least as much as mAP does. Cost is ~10–15 min per seed for all models × 2 CAM methods over the 440-image test split; **~45 min total**. Affordable, and without it an Axis B difference is unfalsifiable.

---
### What the detection results already set up

yolov8s smoke test, per-class mAP@0.5:

| | mAP@0.5 |
|---|---|
| Aortic enlargement | **0.945** |
| Cardiomegaly | **0.912** |
| *other 12, mean* | **0.207** |

Both leaders are **anatomically anchored** — the aorta and heart sit in the same place in every image, so a model can score by learning *position* rather than pathology.

**That is this notebook's sharpest question.** If CAMs for Aortic enlargement attend to the aorta, the model learned anatomy. If they're diffuse or centre-weighted while detection stays correct, the 0.945 is position priors — and only saliency measurement can tell the difference.

---
**Prerequisite:** Add Data → "Notebook Output Files" → "Your Work" → attach **both** notebook 01 (dataset) and notebook 02 (weights). **Accelerator:** `GPU T4 x2`, `device=0`.

In [ ]:
!pip install -q ultralytics grad-cam

REPO = "/kaggle/working/repo"
!rm -rf {REPO} && git clone -q https://github.com/ShMazumder/vinbigdata-chest-x-ray.git {REPO}

import sys
sys.path.insert(0, REPO)

from pathlib import Path
import pandas as pd, numpy as np
from ultralytics import YOLO

from src import config as C
from src.evaluation import xai
from src.utils.run_logger import capture_gpu

!cd {REPO} && git rev-parse --short HEAD
capture_gpu(strict=True)

In [ ]:
DATA_YAML = C.find_data_yaml()

# NOT DATA_YAML.parent -- find_data_yaml() may return a repaired yaml written
# to /kaggle/working while the data stays on the read-only mount.
DATA_ROOT = C.dataset_root(DATA_YAML)
TEST_IMG = DATA_ROOT / "images" / "test"
TEST_LBL = DATA_ROOT / "labels" / "test"
assert TEST_IMG.is_dir(), f"no test images at {TEST_IMG}"
print(f"test split: {len(list(TEST_IMG.glob('*.jpg')))} images")

In [ ]:
# All (model, seed) runs. find_weights() would collapse to one per model.
import re

runs = {}
for p in Path("/kaggle/input").glob("**/weights/best.pt"):
    name = p.parent.parent.name              # yolo26s_512_e40_seed0
    m = re.match(r"(\w+?)_\d+_e\d+_seed(\d+)$", name)
    if m and m.group(1) in C.MODELS:
        runs[(m.group(1), int(m.group(2)))] = str(p)

assert runs, "attach notebook 02's output"
print(f"{len(runs)} runs found:")
for k in sorted(runs):
    print(f"  {k[0]} seed={k[1]}")

MODEL_KEYS = sorted({k for k, _ in runs})
SEEDS_FOUND = sorted({s for _, s in runs})

## 1. Target layer check — DO THIS FIRST

> ⚠️ **The fragile part.** Target-layer selection differs across v8/v11/YOLO26. `pick_target_layer()` takes the last `Conv2d`, which may land *inside* the detection head rather than before it.
>
> **Verify the printed layer is pre-head.** A CAM from the wrong layer produces plausible-looking garbage — it won't error, it'll just be wrong, and every number downstream inherits it.

In [ ]:
probe = {k: YOLO(runs[(k, SEEDS_FOUND[0])]) for k in MODEL_KEYS}
for k, m in probe.items():
    print(f"\n--- {k} ---")
    xai.pick_target_layer(m)

## 2. Qualitative check (D7)

Look at CAMs overlaid on GT boxes **before** the batch job. Fusion runs 1 and 2 both passed `verify()` while producing boxes stacked 8 deep — the cheap visual pass catches what automated gates don't.

In [ ]:
import cv2, matplotlib.pyplot as plt, matplotlib.patches as patches
from src.evaluation.metrics import ground_truth_from_yolo

ref = "yolo26s" if "yolo26s" in probe else MODEL_KEYS[0]
gts = ground_truth_from_yolo(TEST_LBL, TEST_IMG)
sample = sorted(TEST_IMG.glob("*.jpg"))[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, p in zip(axes.ravel(), sample):
    img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    h, w = img.shape
    cam = xai.compute_cam(probe[ref], img, method="eigencam", imgsz=C.IMGSZ)
    ax.imshow(cv2.resize(img, (C.IMGSZ, C.IMGSZ)), cmap="gray")
    ax.imshow(cam, cmap="jet", alpha=0.45)
    for _, r in gts[gts.image_id == p.stem].iterrows():
        sx, sy = C.IMGSZ / w, C.IMGSZ / h
        ax.add_patch(patches.Rectangle((r.x1*sx, r.y1*sy), (r.x2-r.x1)*sx,
                                       (r.y2-r.y1)*sy, fill=False,
                                       edgecolor="lime", linewidth=2))
    ax.set_title(p.stem, fontsize=9); ax.axis("off")
plt.suptitle(f"{ref} — EigenCAM vs radiologist boxes", y=1.01)
plt.tight_layout()

### 2b. The anchored-class check

Aortic enlargement scored **0.945** after five epochs. Suspiciously easy for a pathology.

**Does the CAM attend to the aorta, or to image centre?** Saliency inside the box → the model learned anatomy. Diffuse or centre-weighted while detection stays correct → position priors, and a concrete demonstration of the accuracy/faithfulness gap this paper is about.

In [ ]:
AORTIC = 0
aortic_imgs = gts[gts.class_id == AORTIC].image_id.unique()[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, img_id in zip(axes.ravel(), aortic_imgs):
    img = cv2.imread(str(TEST_IMG / f"{img_id}.jpg"), cv2.IMREAD_GRAYSCALE)
    h, w = img.shape
    cam = xai.compute_cam(probe[ref], img, method="eigencam", imgsz=C.IMGSZ)
    boxes = gts[(gts.image_id == img_id) & (gts.class_id == AORTIC)][
        ["x1", "y1", "x2", "y2"]].to_numpy()
    ebpg = xai.energy_pointing_game(cam, boxes, (w, h), C.IMGSZ)

    ax.imshow(cv2.resize(img, (C.IMGSZ, C.IMGSZ)), cmap="gray")
    ax.imshow(cam, cmap="jet", alpha=0.45)
    for x1, y1, x2, y2 in boxes:
        sx, sy = C.IMGSZ / w, C.IMGSZ / h
        ax.add_patch(patches.Rectangle((x1*sx, y1*sy), (x2-x1)*sx, (y2-y1)*sy,
                                       fill=False, edgecolor="lime", linewidth=2))
    ax.set_title(f"EBPG={ebpg:.3f}", fontsize=10); ax.axis("off")
plt.suptitle("Aortic enlargement (mAP 0.945) — anatomy or position?", y=1.01)
plt.tight_layout()

## 3. Batch metrics across all runs (D8, ~45 min)

| Metric | What it measures |
|---|---|
| `pointing_game` | CAM argmax inside GT box? Coarse but standard. |
| `ebpg` | Fraction of CAM **energy** inside GT box. **Lead with this** — uses the whole map, so it separates "peaked correctly by luck" from "consistently attends". |
| `cam_box_iou` | IoU of 90th-percentile-thresholded CAM vs GT |

Saved incrementally — an interrupted session doesn't lose completed runs.

In [ ]:
C.RUNS_DIR.mkdir(parents=True, exist_ok=True)
out_csv = C.RUNS_DIR / "xai_metrics.csv"

done = set()
if out_csv.exists():
    prev = pd.read_csv(out_csv)
    done = set(map(tuple, prev[["model", "seed", "method"]].drop_duplicates().values))
    print(f"resuming, {len(done)} (model,seed,method) combos already done")

all_xai = [prev] if done else []
for (key, seed), w in sorted(runs.items()):
    model = YOLO(w)
    for method in C.XAI_METHODS:
        if (key, seed, method) in done:
            continue
        print(f"\n=== {key} seed={seed} / {method} ===")
        df = xai.evaluate_xai(model, TEST_IMG, TEST_LBL, method=method,
                              imgsz=C.IMGSZ, n_images=C.XAI_N_IMAGES)
        df["model"], df["seed"] = key, seed
        all_xai.append(df)
        pd.concat(all_xai, ignore_index=True).to_csv(out_csv, index=False)

xai_df = pd.concat(all_xai, ignore_index=True)
print(f"\n{len(xai_df)} rows, {xai_df.groupby(['model','seed']).ngroups} runs")

## 4. Axis A — small vs large targets

**Hypothesis:** explanation faithfulness degrades on small targets *faster than mAP does*.

Detection already sets it up — Calcification **0.097**, Other lesion **0.045**, against Aortic enlargement **0.945**. If faithfulness falls off faster than that, **the model looks right while explaining wrong**, precisely where a radiologist most needs the explanation.

In [ ]:
eig = xai_df[xai_df.method == "eigencam"]

# Mean over seeds, per model, per target-size group.
grouped = eig.assign(group=eig.class_id.map(
    lambda c: "small" if c in C.SMALL_TARGET_CLASSES
    else ("large" if c in C.LARGE_TARGET_CLASSES else "other")))

per_seed = grouped.groupby(["model", "seed", "group"])["ebpg"].mean().reset_index()
axis_a = per_seed.groupby(["model", "group"])["ebpg"].agg(["mean", "std"]).round(4)
axis_a

In [ ]:
sub = eig.copy()
sub["area_bin"] = pd.qcut(sub.box_area_frac, 5, labels=["XS","S","M","L","XL"])
per_seed_bin = sub.groupby(["model", "seed", "area_bin"], observed=False)["ebpg"].mean().reset_index()
pivot = per_seed_bin.pivot_table(index="area_bin", columns="model",
                                 values="ebpg", aggfunc="mean", observed=False)
err = per_seed_bin.pivot_table(index="area_bin", columns="model",
                               values="ebpg", aggfunc="std", observed=False)

pivot.plot(marker="o", yerr=err, capsize=3, figsize=(8,5),
           title="EBPG vs object size (mean ± std over seeds)")
plt.ylabel("Energy-based pointing game"); plt.grid(alpha=.3)
pivot.round(4)

In [ ]:
# Per-class faithfulness beside per-class detection -- the paper's core table.
per_class_xai = (eig.groupby(["model", "seed", "class"])["ebpg"].mean()
                 .reset_index()
                 .groupby(["model", "class"])["ebpg"].agg(["mean", "std"]).round(4))
per_class_xai.unstack(0)

## 5. Axis B — NMS-free vs NMS

**This ties the paper's halves together.** Does YOLO26's native NMS-free head produce more faithful explanations than v8/v11?

The seed spread decides whether any difference is claimable — same test applied to detection in notebook 02.

In [ ]:
metrics3 = ["pointing_game", "ebpg", "cam_box_iou"]
per_seed_b = xai_df.groupby(["model", "seed", "method"])[metrics3].mean().reset_index()
axis_b = per_seed_b.groupby(["model", "method"])[metrics3].agg(["mean", "std"]).round(4)
print(axis_b)

# Is the EBPG ranking supported, or inside seed noise?
e = per_seed_b[per_seed_b.method == "eigencam"]
means = e.groupby("model").ebpg.mean()
stds = e.groupby("model").ebpg.std().fillna(0)
gap, noise = means.max() - means.min(), stds.mean()
print(f"\nEBPG between-model gap: {gap:.4f} | mean within-model std: {noise:.4f}")
if gap < noise:
    print("⚠ Inside seed noise -- report models as indistinguishable on EBPG.")
elif gap < 2 * noise:
    print("⚠ Under 2x seed noise -- weak evidence, hedge the claim.")
else:
    print("✓ Exceeds 2x seed noise -- Axis B ranking is defensible.")

## 6. Axis C — accuracy/faithfulness decoupling

Does the highest-mAP model give the best explanations?

**A "no" is the most publishable outcome available.** It says accuracy and explanation quality are separate axes — exactly the argument for measuring the second rather than assuming it follows the first.

Note this survives even if *both* rankings are inside seed noise: "neither accuracy nor faithfulness separates these architectures, but faithfulness varies sharply by lesion size" is still a clean result, and Axis A carries it.

In [ ]:
import json

cands = list(Path("/kaggle/input").glob("**/eval_summary.json"))
assert cands, "attach notebook 02's output"
ev = json.loads(cands[0].read_text())

acc = (pd.DataFrame([{"model": v["model"], "seed": v["seed"],
                      "map40": v["map40"], "map50": v["map50"]}
                     for v in ev.values()])
       .groupby("model")[["map40", "map50"]].agg(["mean", "std"]).round(4))
acc.columns = ["_".join(c) for c in acc.columns]

fai = e.groupby("model")[["ebpg", "pointing_game"]].agg(["mean", "std"]).round(4)
fai.columns = ["_".join(c) for c in fai.columns]

combined = acc.join(fai)
combined["rank_mAP"] = combined.map40_mean.rank(ascending=False).astype(int)
combined["rank_EBPG"] = combined.ebpg_mean.rank(ascending=False).astype(int)
combined["DECOUPLED"] = combined.rank_mAP != combined.rank_EBPG
combined

## 7. Causal faithfulness — optional, D9 only if ahead

Deletion/insertion AUC needs ~40× the forward passes. **Subsample, seed 0 only.**

A saliency map that looks convincing but has flat deletion AUC isn't explaining the model — precisely the failure MedFocus / MedGround-Bench reported for standard attribution on VinDr.

In [ ]:
RUN_CAUSAL = False   # flip only if D9 is on schedule

if RUN_CAUSAL:
    causal = xai.evaluate_xai(probe[ref], TEST_IMG, TEST_LBL,
                              method="eigencam", imgsz=C.IMGSZ,
                              n_images=50, with_causal=True, causal_subsample=50)
    causal.to_csv(C.RUNS_DIR / "xai_causal.csv", index=False)
    print(causal[["deletion_auc", "insertion_auc"]].mean())

## 8. Export for the paper (D10)

All tables carry mean±std over seeds. Feed into the `latex-results-table` skill for booktabs formatting.

In [ ]:
combined.to_csv(C.RUNS_DIR / "table_main.csv")
axis_a.to_csv(C.RUNS_DIR / "table_axis_a_groups.csv")
pivot.to_csv(C.RUNS_DIR / "table_axis_a_bins.csv")
axis_b.to_csv(C.RUNS_DIR / "table_axis_b.csv")
per_class_xai.to_csv(C.RUNS_DIR / "table_per_class_xai.csv")

print("tables written. Write-up reminders:")
print("  - report BOTH mAP@0.4 and mAP@0.5, all as mean±std over 3 seeds")
print("  - positive-only protocol in the ABSTRACT; published numbers get a")
print("    protocol column and never share a column with ours")
print("  - two anchored classes carry the headline mAP -- report per-class")
print("  - if a gap is inside seed noise, say indistinguishable; don't rank")
print("  - limitations: 512px, s-scale, single fold, positive-only, 40 epochs,")
print("    95.5% of labels from 3 radiologists, WBF@0.25 fusion, T4")
print("  - Atelectasis n=22 / Pneumothorax n=10: per-class AP with n")